# 📘 Notebook 16: Text from the Web

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module F: Language Models in Practice · Notebook 2 of 4 in this module · (16 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Read the structure of an **HTML** page and say why extracting text from it needs a parser
- Extract titles, paragraphs, links and tables with **BeautifulSoup**
- Fetch a page with **requests**, and handle the ways in which a request can fail
- Prefer an **API** to scraping when one exists, and use one
- Check **robots.txt**, limit your request rate, and identify your program
- Reason about the **legal and ethical** side of collecting text: copyright, personal data, bias
- Build a small corpus from the web and analyse it with the tools of earlier notebooks

> **Prerequisites:** Notebooks 02, 03 and 08.
>
> 🔭 **Why this notebook matters:** every corpus in this course so far arrived ready-made. In real projects the text has to be found and collected first, and most of it lives on the web. The large language models of this module learned from text gathered in exactly this way, at enormous scale. Knowing how that is done, and what can go wrong, is part of understanding both what those models know and where their knowledge may be skewed. It is also a practical skill you will need the first time a project has no dataset.

> ℹ️ **Setup note.** `requests` and `beautifulsoup4` are already installed in Colab. Sections 1 and 2 work without an internet connection. From Section 3 onwards the cells fetch pages from Wikipedia, so the exact output will change as those pages are edited.

> 📦 **Libraries used in this notebook.**
>
> - **Requests** is the most widely used Python library for HTTP, the protocol of the web. It sends a request to an address and gives back the response.
> - **Beautiful Soup** (installed as `beautifulsoup4`, imported as `bs4`) reads HTML and lets you navigate its elements. It has existed since 2004 and is forgiving of the badly formed pages that are common on the web.
> - **NLTK** supplies a stop-word list and **scikit-learn** (Notebook 08) the TF-IDF vectoriser.
>
> The cell below installs what is needed. In Google Colab these libraries are already present and the cell finishes in a moment. On your own computer it may take a minute the first time.

In [ ]:
%pip install -q requests beautifulsoup4 nltk scikit-learn

In [ ]:
import json
import re
import time
from collections import Counter

import requests
from bs4 import BeautifulSoup

## 1. What a web page really is

### Intuition first
A web page looks like formatted text and pictures. What your browser actually receives is one long **string**, in which the text you see is wrapped in instructions. The instructions say what each piece **is**: a heading, a paragraph, a link, a row of a table. This is **HTML**, the HyperText Markup Language.

The instructions are written as **tags** in angle brackets. An opening tag such as `<p>` and a closing tag such as `</p>` enclose an **element**. Elements sit inside one another, like the constituents of a sentence in Notebook 05, so a page is a **tree**. Tags may carry **attributes**, such as the address a link points to.

Here is a small page, held in a Python string so that we can work without a connection.

In [ ]:
page = """
<html>
  <head>
    <title>Harbour Books: New Arrivals</title>
    <script>var visitors = 1024; trackVisit("<p>not text</p>");</script>
  </head>
  <body>
    <div class="menu"><a href="/">Home</a> | <a href="/about">About us</a></div>
    <h1>New arrivals this week</h1>
    <p class="intro">Three books reached our shelves on <b>Monday</b>.</p>
    <table id="books">
      <tr><th>Title</th><th>Author</th><th>Price</th></tr>
      <tr><td>The Salt Road</td><td>M. Ortega</td><td>12.50</td></tr>
      <tr><td>A Winter Atlas</td><td>L. Brandt</td><td>18.00</td></tr>
      <tr><td>Paper Lanterns</td><td>S. Okafor</td><td>9.90</td></tr>
    </table>
    <p>Read our <a href="/reviews/salt-road">review of The Salt Road</a> or
       visit <a href="https://example.org/brandt">the author's page</a>.</p>
    <div class="footer">Contact: info@harbourbooks.example &copy; 2025</div>
  </body>
</html>
"""
print(len(page), "characters")

### Why not a regular expression?
In Notebook 02 we removed HTML tags with the pattern `<[^>]+>`. Let us try it on this page.

In [ ]:
stripped = re.sub(r"<[^>]+>", " ", page)
print(re.sub(r"\s+", " ", stripped).strip())

The tags are gone, and the result is a mess. The JavaScript code is now mixed in with the text, and so is the menu. We can no longer tell the heading from a paragraph, which words were links, or which numbers were prices. All the **structure**, which is exactly what tells us what each piece of text is, has been thrown away.

There is a deeper reason too, which you met in Notebook 04. HTML elements nest inside one another to any depth. Nested structure is beyond what regular expressions can describe. What we need is a **parser**.

## 2. Parsing HTML with BeautifulSoup

### Formal definition
An HTML **parser** reads the string and builds the tree of elements. **BeautifulSoup** is the most widely used Python library for navigating that tree. Its essential tools are few:

| Call | Returns |
|---|---|
| `soup.find("p")` | the first `<p>` element |
| `soup.find_all("p")` | a list of all `<p>` elements |
| `soup.find("table", id="books")` | the first element with that tag and attribute |
| `element.get_text()` | the visible text inside an element |
| `element["href"]` | the value of an attribute |

In [ ]:
soup = BeautifulSoup(page, "html.parser")

print("Title:  ", soup.title.get_text())
print("Heading:", soup.find("h1").get_text())
print()
for paragraph in soup.find_all("p"):
    print("Paragraph:", " ".join(paragraph.get_text().split()))

We asked for paragraphs and received paragraphs: no script, no menu, no footer. The bold tag inside the first paragraph caused no trouble, since `get_text()` collects the text of everything nested inside.

### Links
A link is an `<a>` element. Its visible text and its destination are different things, and for building a corpus we usually want both.

In [ ]:
for link in soup.find_all("a"):
    print(f"{link.get_text():<28} -> {link['href']}")

Some destinations are complete addresses and others are **relative**: `/about` means "the page `about` on this same site". To follow a relative link you must join it to the address of the page it came from, which `urllib.parse.urljoin` does.

In [ ]:
from urllib.parse import urljoin

base = "https://harbourbooks.example/new"
for link in soup.find_all("a"):
    print(urljoin(base, link["href"]))

### Tables
A table is rows (`<tr>`) of cells (`<td>`, or `<th>` for headers). Walking the rows turns it into a list of dictionaries, the form in which data is easiest to use.

In [ ]:
table = soup.find("table", id="books")
rows = table.find_all("tr")

headers = [cell.get_text() for cell in rows[0].find_all("th")]
books = []
for row in rows[1:]:
    values = [cell.get_text() for cell in row.find_all("td")]
    books.append(dict(zip(headers, values)))

for book in books:
    print(book)
print("Total:", sum(float(book["Price"]) for book in books))

### Selecting by class
Pages mark the role of an element with a `class` attribute. Here the introduction has `class="intro"` and the footer `class="footer"`. Selecting by class is the usual way to pick the content out of a page and leave the navigation behind.

In [ ]:
print(soup.find("p", class_="intro").get_text())
print(soup.find("div", class_="footer").get_text())

> ⚠️ **Common pitfalls**
>
> - Assuming an element exists. `soup.find(...)` returns `None` when nothing matches, and calling `.get_text()` on `None` raises an error. Check first.
> - Writing a scraper that depends on fine details of one page's layout. Sites are redesigned without notice, and a scraper built on such details then fails silently, returning empty results.
> - Expecting to find everything in the HTML. Many modern sites load their content with JavaScript after the page arrives, so the HTML you download contains only an empty frame. Such sites nearly always have an API behind them, which is the better thing to look for.

## 3. Fetching a page

### Intuition first
Your program can do what a browser does: send a **request** to a server for an address, and receive a **response**. The `requests` library makes this one line. Around that line there are three courtesies and precautions that a well-behaved program always observes.

1. **Say who you are.** The `User-Agent` header identifies the program making the request. Many sites refuse requests that do not identify themselves.
2. **Set a timeout.** Otherwise a server that never answers will make your program wait for ever.
3. **Check the status.** The response carries a status code: `200` means success, `404` that the page does not exist, `403` that access is refused, `429` that you are asking too often, and codes from `500` that the server itself has failed.

The function below does all three. When the server answers `429` it waits a few seconds and tries again, a little more patiently each time.

In [ ]:
HEADERS = {"User-Agent": "nlp-course-notebook/1.0 (educational use)"}

def fetch(url, attempts=3):
    """Return the text of a page, or None if the request did not succeed."""
    for attempt in range(attempts):
        try:
            response = requests.get(url, headers=HEADERS, timeout=20)
        except requests.RequestException as problem:
            print("Request failed:", type(problem).__name__)
            return None
        if response.status_code == 429 and attempt < attempts - 1:
            time.sleep(5 * (attempt + 1))          # we were asked to slow down: wait, then try again
            continue
        if response.status_code != 200:
            print("Status", response.status_code, "for", url)
            return None
        return response.text

html = fetch("https://en.wikipedia.org/wiki/Natural_language_processing")
print(len(html), "characters of HTML")
print(html[:300])

Several hundred thousand characters, and the visible article is a small part of them. Now the parser earns its keep.

In [ ]:
soup = BeautifulSoup(html, "html.parser")
print(soup.title.get_text())

paragraphs = []
for paragraph in soup.find_all("p"):
    text = paragraph.get_text()
    text = re.sub(r"\[\d+\]", "", text)          # remove reference marks such as [12]
    text = " ".join(text.split())                # tidy the whitespace
    if len(text) > 80:                           # skip empty and very short paragraphs
        paragraphs.append(text)

print(len(paragraphs), "paragraphs\n")
print(paragraphs[0])

Three lines of cleaning, each a decision. We removed the bracketed reference numbers with a regular expression, tidied the whitespace, and dropped fragments too short to be real paragraphs. This is the proper division of labour between the two tools: the **parser** finds the elements, and **regular expressions** clean the text inside them.

With clean text in hand, everything from the earlier notebooks applies. For instance, the most frequent content words.

In [ ]:
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
article = " ".join(paragraphs)
words = [word for word in re.findall(r"[a-z]+", article.lower()) if word not in stop_words and len(word) > 2]

print(f"{len(words):,} content words")
print(Counter(words).most_common(12))

## 4. When there is an API, use it

### Intuition first
Scraping takes a page designed for human eyes and tries to recover the data hidden in its layout. Many sites offer a much better door: an **API** (application programming interface), an address that returns the data itself, cleanly structured, usually as **JSON**. Using the API is like asking the librarian for a book, where scraping is like photographing the shelves.

Wikipedia has one. This address returns a short summary of an article.

In [ ]:
def wikipedia_summary(title):
    """Return the summary of a Wikipedia article as a dictionary, or None."""
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + title.replace(" ", "_")
    text = fetch(url)
    if text is None:
        return None
    return json.loads(text)

summary = wikipedia_summary("Natural language processing")
print(list(summary.keys()))
print()
print(summary["title"])
print(summary["extract"])

No parsing, no cleaning, no guessing which element holds the content. The reply is a dictionary with named fields. An API is also more **stable** than a page layout, and it is the route the site's owners intend programs to take.

Our `fetch` function handles failure, so a request for a page that does not exist is reported and does not crash the program.

In [ ]:
print(wikipedia_summary("This article certainly does not exist 12345"))

> 🧠 **Order of preference when you need text from a site:** first look for a ready-made **dataset** or a downloadable dump. Next, an official **API**. Scrape pages only when neither exists, and only when you are allowed to.

## 5. Being a good guest

A person visits a few pages a minute. A program can request thousands a second, which can slow a small site to a halt and, in effect, attack it. Collecting text responsibly means following some conventions.

### robots.txt
Most sites publish a file called `robots.txt` at their top level. It states which parts of the site automated programs may visit. Python can read it for us.

In [ ]:
from urllib import robotparser

rules = robotparser.RobotFileParser()
rules.parse(fetch("https://en.wikipedia.org/robots.txt").splitlines())

for url in ["https://en.wikipedia.org/wiki/Natural_language_processing",
            "https://en.wikipedia.org/w/index.php?search=language",
            "https://en.wikipedia.org/wiki/Special:Random"]:
    print(f"{'allowed' if rules.can_fetch('nlp-course-notebook', url) else 'NOT allowed':<12} {url}")

Ordinary articles may be fetched. The search pages and the special pages may not: they are costly for the server to produce, and of no use to a program that only wants to read articles.

Exercise 4 will show you something that looks awkward at first: the same file also lists the address of the **API** we used in Section 4 as off limits. There is no contradiction. `robots.txt` is addressed to **crawlers**, the programs that wander from link to link to index a whole site, and the site does not want them indexing API output. A program that calls the API on purpose is governed by a different document, the API's own published rules, which for Wikipedia ask you to identify your program in the `User-Agent` header and to keep your request rate modest. The lesson is that `robots.txt` is one of the documents to read, and not the only one.

### Go slowly, and do not ask twice
Two more habits complete the picture: **wait** between requests, and **keep** what you have fetched, so that you never request the same page twice.

In [ ]:
cache = {}

def polite_summary(title, delay=1.0):
    if title in cache:
        return cache[title]                 # already fetched: no new request
    time.sleep(delay)                       # wait before each new request
    summary = wikipedia_summary(title)
    if summary is not None:
        cache[title] = summary              # remember successes only, so that a failure can be retried
    return summary

started = time.perf_counter()
polite_summary("Zipf's law")
first = time.perf_counter() - started

started = time.perf_counter()
polite_summary("Zipf's law")
second = time.perf_counter() - started

print(f"First call:  {first:.2f} seconds")
print(f"Second call: {second:.5f} seconds (answered from the cache)")

## 6. Law and ethics

That a page can be downloaded does not mean that it may be, nor that its text can be used for any purpose. This is not a legal handbook, and the rules differ between countries, but four questions should be asked before any collection of text.

**1. Am I allowed to access it?** Read the site's terms of use as well as `robots.txt`. Never work around a login, a paywall or a technical block.

**2. Who owns the text?** Text on the web is normally protected by **copyright**, even when it is free to read. Analysing it for research is treated differently, in many jurisdictions, from republishing it or building a commercial product on it. Wikipedia's text is under an open licence that permits reuse with attribution. A newspaper's articles are not.

**3. Does it contain personal data?** Posts, reviews, forum messages and comments are written by identifiable people. In the European Union the **GDPR** applies to personal data even when it is publicly visible. Collect only what you need, remove names and contact details where you can (the `anonymise` function of Notebook 02 is a start), store the data securely, and never try to work out who an anonymous author is.

**4. Could collecting or using it cause harm?** Think about what the data will make possible. Health forums, political opinions and messages from young people deserve particular care, whatever the law allows.

### What the web does to a model
There is one more consideration, about the **quality** of what you gather. Text from the web is not a fair sample of human language or of human beings.

- A few languages, English above all, are hugely over-represented.
- Some groups write far more online than others, and some topics attract far more text.
- The web contains errors, spam, propaganda, and by now a growing amount of text written by machines.

A model trained on web text inherits all of this, as we saw with the embeddings of Notebook 10. When a language model seems to "know" something, what it has is a compressed record of what was written on the pages it was trained on, by the people who wrote them.

| Before collecting | Ask |
|---|---|
| Access | Do the terms of use and `robots.txt` permit it? Is there an API or a dataset? |
| Ownership | What is the licence? What will I do with the text? |
| People | Is there personal data? Do I need it? How will I protect it? |
| Load | Am I identifying myself, waiting between requests, and caching? |
| Quality | Whose voices are in this text, and whose are missing? |

## 7. From pages to a corpus

Let us put it together. We collect the summaries of twelve articles on four themes, politely, and then ask the tools of Notebook 08 whether the texts group by theme.

In [ ]:
topics = {
    "animals":   ["Lion", "Elephant", "Dolphin"],
    "languages": ["Python (programming language)", "Java (programming language)", "JavaScript"],
    "music":     ["Violin", "Piano", "Guitar"],
    "space":     ["Mars", "Jupiter", "Saturn"],
}

corpus = []
for theme, titles in topics.items():
    for title in titles:
        summary = polite_summary(title, delay=1.0)
        if summary is not None:
            corpus.append((theme, title, summary["extract"]))

print(len(corpus), "documents collected\n")
for theme, title, text in corpus[:3]:
    print(f"[{theme}] {title}: {text[:110]}...")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectoriser = TfidfVectorizer(stop_words="english")
matrix = vectoriser.fit_transform([text for theme, title, text in corpus])
similarity = (matrix @ matrix.T).toarray()

print("For each document, the most similar other document:\n")
agree = 0
for i, (theme, title, text) in enumerate(corpus):
    similarity[i, i] = -1
    nearest = int(similarity[i].argmax())
    same = corpus[nearest][0] == theme
    agree += same
    print(f"{title[:30]:<32} -> {corpus[nearest][1][:30]:<32} {'same theme' if same else 'DIFFERENT THEME'}")

print(f"\nNearest neighbour has the same theme for {agree} of {len(corpus)} documents.")

From a list of titles to a small analysed corpus in about twenty lines, fetched considerately through an API. The pattern scales directly: more titles, a longer delay, a cache saved to a file, and you have the beginning of a real dataset.

---
## ✏️ Exercises

### Exercise 1 (Links, inside and out)
Using the `page` string of Section 1, print two lists: the **internal** links (those leading to the same site) and the **external** ones, as full addresses. Assume the page was fetched from `https://harbourbooks.example/new`. (Hint: after `urljoin`, an internal link starts with `https://harbourbooks.example`.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
shop = BeautifulSoup(page, "html.parser")
site = "https://harbourbooks.example"

internal, external = [], []
for link in shop.find_all("a"):
    address = urljoin(site + "/new", link["href"])
    if address.startswith(site):
        internal.append(address)
    else:
        external.append(address)

print("Internal:", internal)
print("External:", external)
```

*A crawler that follows links must make this distinction. It usually stays within one site, and it must turn every relative link into a full address before it can follow it.*
</details>

### Exercise 2 (Data from a table)
From the same `page`, build the list of books again and print the title and author of the **cheapest** book and the **average** price. Convert the prices to numbers.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
shop = BeautifulSoup(page, "html.parser")
rows = shop.find("table", id="books").find_all("tr")

catalogue = []
for row in rows[1:]:
    title, author, price = [cell.get_text() for cell in row.find_all("td")]
    catalogue.append({"title": title, "author": author, "price": float(price)})

cheapest = min(catalogue, key=lambda book: book["price"])
print("Cheapest:", cheapest["title"], "by", cheapest["author"])
print("Average price:", round(sum(book["price"] for book in catalogue) / len(catalogue), 2))
```

*Everything read from a page is a string. Converting to the right type, and checking that the conversion makes sense, is part of extraction.*
</details>

### Exercise 3 (Clean and anonymise)
Write `clean_text(element)` that takes a BeautifulSoup element and returns its text with the whitespace tidied, any e-mail address replaced by `[EMAIL]`, and the copyright sign and year removed. Apply it to the footer of `page`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def clean_text(element):
    text = " ".join(element.get_text().split())
    text = re.sub(r"\S+@\S+", "[EMAIL]", text)
    text = re.sub(r"©\s*\d{4}", "", text)
    return text.strip()

shop = BeautifulSoup(page, "html.parser")
print(clean_text(shop.find("div", class_="footer")))
```

*The parser turned the code `&copy;` into the real © character for us. Removing contact details at the moment of collection, before anything is stored, is the safest habit when a page may contain personal data.*
</details>

### Exercise 4 (Read the rules)
Using the `rules` object of Section 5, check whether a program may fetch each of these Wikipedia addresses: `/wiki/Python_(programming_language)`, `/wiki/Special:Search`, `/w/api.php` and `/api/rest_v1/page/summary/Lion`. (Add `https://en.wikipedia.org` in front.) How do you reconcile the result with our use of the summary API?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for path in ["/wiki/Python_(programming_language)", "/wiki/Special:Search", "/w/api.php", "/api/rest_v1/page/summary/Lion"]:
    allowed = rules.can_fetch("nlp-course-notebook", "https://en.wikipedia.org" + path)
    print(f"{'allowed' if allowed else 'NOT allowed':<12} {path}")
```

*The API addresses are listed as off limits, as Section 5 warned. The file speaks to crawlers that index a site by following links. Deliberate use of the API is covered by the API's own usage rules, which we follow by identifying our program and asking slowly. When two documents seem to disagree, read both, and if you are still unsure, ask the site. Note also that robots.txt is a statement of the owner's wishes and not a technical barrier: nothing stops a program from ignoring it. Respecting it is a matter of conduct.*
</details>

### Exercise 5 (Keywords of an article)
Write `article_keywords(title, how_many=10)` that fetches the summary of a Wikipedia article through `polite_summary` and returns its most frequent content words (no stop words, at least three letters). Try it on `"Photosynthesis"` and `"Olympic Games"`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def article_keywords(title, how_many=10):
    summary = polite_summary(title)
    if summary is None:
        return []
    found = [word for word in re.findall(r"[a-z]+", summary["extract"].lower())
             if word not in stop_words and len(word) > 2]
    return [word for word, count in Counter(found).most_common(how_many)]

for title in ["Photosynthesis", "Olympic Games"]:
    print(title, "->", article_keywords(title))
```

*The function returns an empty list when the page cannot be fetched. Code that works with the web must expect failure at every request: pages move, servers are busy, connections drop.*
</details>

### Exercise 6 (A small, careful crawler, a little harder)
Write `collect(titles, delay=1.0)` that fetches the summaries of a list of article titles and returns a dictionary from title to text. It should wait between requests, skip titles it cannot fetch without stopping, and report at the end how many succeeded and how many failed. Test it with a list that includes one title that does not exist. Then save the result to a file `corpus.json` and read it back.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def collect(titles, delay=1.0):
    collected = {}
    failed = []
    for title in titles:
        summary = polite_summary(title, delay)
        if summary is None or "extract" not in summary:
            failed.append(title)
        else:
            collected[title] = summary["extract"]
    print(f"{len(collected)} collected, {len(failed)} failed: {failed}")
    return collected

texts = collect(["Honey bee", "Volcano", "No such article qwertyuiop", "Chess"])

with open("corpus.json", "w", encoding="utf-8") as file:
    json.dump(texts, file, ensure_ascii=False, indent=1)

with open("corpus.json", encoding="utf-8") as file:
    restored = json.load(file)
print(list(restored), "| same content:", restored == texts)
```

*Saving what you have collected means the site is asked only once, however many times you rerun your analysis. Note the `encoding="utf-8"`, exactly as Notebook 01 advised.*
</details>

## 🔑 Key takeaways

- A web page is a string of **HTML**: text wrapped in nested, labelled elements. The structure tells you what each piece of text is.
- Use a **parser** such as BeautifulSoup to find elements, and **regular expressions** to clean the text inside them.
- When fetching: **identify** your program, set a **timeout**, check the **status code**, and expect failure.
- Prefer a **dataset** or an **API** to scraping. APIs return clean, structured, stable data.
- Respect **robots.txt** and the terms of use, **wait** between requests, and **cache** what you fetch.
- Being able to download text does not make it yours to use. Consider **copyright**, **personal data** and possible **harm**.
- Web text is a **biased sample** of language and of people, and models trained on it inherit that bias.

---
**Next:** *Notebook 17: Summarisation*, where we take long texts such as the articles collected here and reduce them to their essentials, first by selecting sentences and then by writing new ones.

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*